# Use the fine tuned banking decision model

This notebook downloads the public Laya checkpoint and runs banking intent decisions without training. It uses the same two stage family and intent taxonomy as the fine tuning notebook.

A GPU runtime is recommended. The model is public, so no Hugging Face token is required.


## 0. Install the inference dependencies

Laya is pinned to the version used to export the checkpoint. The environment variable prevents Transformers from importing TensorFlow in Colab.


In [ ]:
%env USE_TF=0
%pip -q install "laya==0.3.11" "huggingface_hub>=0.30" "transformers>=4.48,<6" "numpy>=1.26,<3" "pandas>=2.2,<3"


## 1. Download the frozen question taxonomy

The model weights and the question wording form one interface. This cell downloads `questions.json` from the repository so inference uses exactly the same families, option order and descriptions as training.


In [ ]:
import json
import urllib.request
from pathlib import Path

QUESTIONS_URL = "https://raw.githubusercontent.com/dimitrisdais/language-aware-decisions-for-banking/main/configs/questions.json"
QUESTIONS_PATH = Path("/content/questions.json")
urllib.request.urlretrieve(QUESTIONS_URL, QUESTIONS_PATH)
QUESTIONS = json.loads(QUESTIONS_PATH.read_text(encoding="utf-8"))
assert len(QUESTIONS["family"]["criteria"]) == 8
assert sum(len(QUESTIONS[key]["criteria"]) for key in QUESTIONS if key.startswith("intent__")) == 77
print("Loaded 8 families and 77 intents")


## 2. Load the published Hugging Face checkpoint

`laya.load` downloads the model on the first run and reuses the local Hugging Face cache afterwards. The checkpoint may emit a temperature warning. Predictions remain available, but confidence and action thresholds must be treated as uncalibrated.


In [ ]:
import laya
import numpy as np
import pandas as pd
import torch

MODEL_ID = "dimitrisdais/laya-banking77-typed-decisions"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
agent = laya.load(MODEL_ID, device=DEVICE)
print("Loaded", MODEL_ID, "on", DEVICE)


## 3. Compose the two decisions into one banking intent distribution

The agent returns a family distribution and one conditional distribution for each family. Multiplying and normalizing them produces comparable scores across all 77 intents. The maximum is called a score rather than confidence because calibration was not completed for the published checkpoint.


In [ ]:
families = list(QUESTIONS["family"]["criteria"])
labels = sorted(
    intent
    for family in families
    for intent in QUESTIONS[f"intent__{family}"]["criteria"]
)

def compose(result):
    answers = result["answers"]
    family_probabilities = answers["family"]["probabilities"]
    probabilities = {
        intent: float(family_probabilities[family])
        * float(answers[f"intent__{family}"]["probabilities"][intent])
        for family in families
        for intent in QUESTIONS[f"intent__{family}"]["criteria"]
    }
    vector = np.asarray([probabilities[label] for label in labels], dtype=float)
    return vector / vector.sum()

def predict_banking_intent(text, top_k=5):
    result = agent.predict({"message": text}, QUESTIONS)
    probabilities = compose(result)
    order = np.argsort(-probabilities)[:top_k]
    family_probabilities = result["answers"]["family"]["probabilities"]
    return {
        "text": text,
        "family": max(family_probabilities, key=family_probabilities.get),
        "intent": labels[int(order[0])],
        "score_uncalibrated": float(probabilities[order[0]]),
        "top_intents": [
            {"intent": labels[int(index)], "score": float(probabilities[index])}
            for index in order
        ],
    }


## 4. Run a single decision

Change the message and rerun the cell. The output contains the selected family, selected intent and the five highest uncalibrated intent scores.


In [ ]:
example = predict_banking_intent("My card payment was declined at the supermarket.")
print(json.dumps(example, indent=2))


## 5. Run a small batch

This loop favors clarity over maximum throughput. For a service or benchmark, use `agent.predict_batch` as shown in the training notebook and measure batch size on the target hardware.


In [ ]:
messages = [
    "The transfer I made yesterday is still pending.",
    "My PIN is blocked and I cannot use the card.",
    "Why did the cash machine charge me a fee?",
    "I moved to a new address and need to update my details.",
]
predictions = [predict_banking_intent(message, top_k=3) for message in messages]
display(pd.DataFrame([
    {
        "message": row["text"], "family": row["family"],
        "intent": row["intent"], "score_uncalibrated": row["score_uncalibrated"],
    }
    for row in predictions
]))


## 6. What this notebook does not prove

These are customer service intent decisions, not lending, fraud, compliance or customer eligibility decisions. The checkpoint should not automate bank actions on the raw score. A real deployment needs calibration, out of distribution testing, human escalation rules, latency measurement, monitoring and governance.
